# RBC-style Analytics Engineering: End-to-End Data Modeling Pipeline (Real-World Example)

This notebook demonstrates an **end-to-end data modeling pipeline** you could discuss for an **Analytics Engineer / Stress Testing** role:

- **Bronze (Raw):** land source-like tables
- **Silver (Staging):** standardize + validate
- **Gold (Core):** canonical entities + **SCD2** + **snapshots**
- **Marts:** **star schema** for analytics
- **Stress outputs:** scenario-keyed results (baseline/adverse)

**Tech used:** `pandas` + `sqlite` to keep it portable (swap for Spark/BigQuery/dbt in production).


In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from datetime import date, datetime, timedelta

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

DB_PATH = "risk_analytics_demo.sqlite"
conn = sqlite3.connect(DB_PATH)


## 1) Generate Synthetic Source Data (Bronze)
We simulate typical upstream feeds used in credit-risk / stress testing analytics:
- Customers (profile + rating)
- Loans (contracts)
- Transactions (payments/drawdowns/fees)
- Macroeconomic scenarios (quarterly)


In [ ]:
rng = np.random.default_rng(7)

n_customers = 200
n_loans = 400
n_tx = 40_000

start_dt = date(2024, 1, 1)
end_dt   = date(2025, 12, 31)

def rand_dates(n, start=start_dt, end=end_dt):
    days = (end - start).days
    return [start + timedelta(days=int(x)) for x in rng.integers(0, days + 1, size=n)]

# --- Customers ---
customer_ids = np.arange(100000, 100000 + n_customers)
industries = ["Retail", "Manufacturing", "Tech", "Energy", "Healthcare", "Finance"]
regions = ["ON", "QC", "BC", "AB", "MB", "NS"]
ratings = ["AAA","AA","A","BBB","BB","B","CCC"]
income_bands = ["<50k","50-100k","100-200k","200k+"]

src_customers = pd.DataFrame({
    "customer_id": customer_ids,
    "legal_name": [f"Customer_{i}" for i in range(n_customers)],
    "industry": rng.choice(industries, size=n_customers),
    "region": rng.choice(regions, size=n_customers),
    "origination_date": rand_dates(n_customers, start=date(2018,1,1), end=date(2023,12,31)),
    "current_rating": rng.choice(ratings, size=n_customers, p=[0.03,0.07,0.15,0.25,0.25,0.18,0.07]),
    "income_band": rng.choice(income_bands, size=n_customers, p=[0.35,0.35,0.2,0.1]),
    "src_updated_at": rand_dates(n_customers, start=date(2025,1,1), end=date(2025,12,31))
})

# --- Loans ---
loan_ids = np.arange(500000, 500000 + n_loans)
products = ["TermLoan","Revolver","Mortgage","AutoLoan"]
currencies = ["CAD","USD"]

src_loans = pd.DataFrame({
    "loan_id": loan_ids,
    "customer_id": rng.choice(customer_ids, size=n_loans),
    "product_type": rng.choice(products, size=n_loans, p=[0.35,0.25,0.25,0.15]),
    "currency": rng.choice(currencies, size=n_loans, p=[0.85,0.15]),
    "origination_date": rand_dates(n_loans, start=date(2019,1,1), end=date(2025,6,30)),
    "maturity_date": rand_dates(n_loans, start=date(2026,1,1), end=date(2035,12,31)),
    "original_principal": rng.integers(5_000, 500_000, size=n_loans).astype(float),
    "interest_rate": np.round(rng.uniform(0.02, 0.12, size=n_loans), 4),
    "src_updated_at": rand_dates(n_loans, start=date(2025,1,1), end=date(2025,12,31))
})

# --- Transactions ---
tx_types = ["PAYMENT","DRAWDOWN","FEE"]
src_transactions = pd.DataFrame({
    "tx_id": np.arange(1, n_tx + 1),
    "loan_id": rng.choice(loan_ids, size=n_tx),
    "tx_date": rand_dates(n_tx, start=start_dt, end=end_dt),
    "tx_type": rng.choice(tx_types, size=n_tx, p=[0.7,0.25,0.05]),
    "amount": np.round(rng.normal(loc=500, scale=1200, size=n_tx), 2)  # will standardize below
})
# Domain rules:
# - DRAWDOWN increases balance (positive)
# - PAYMENT decreases balance (negative)
# - FEE increases balance slightly (positive small)
src_transactions.loc[src_transactions["tx_type"]=="DRAWDOWN","amount"] = np.abs(src_transactions.loc[src_transactions["tx_type"]=="DRAWDOWN","amount"]) + 100
src_transactions.loc[src_transactions["tx_type"]=="PAYMENT","amount"]  = -np.abs(src_transactions.loc[src_transactions["tx_type"]=="PAYMENT","amount"])
src_transactions.loc[src_transactions["tx_type"]=="FEE","amount"]      = np.abs(src_transactions.loc[src_transactions["tx_type"]=="FEE","amount"]) * 0.2

# --- Macro scenarios (quarterly) ---
quarters = pd.period_range("2024Q1","2025Q4", freq="Q")
scenarios = ["BASELINE","ADVERSE","SEVERELY_ADVERSE"]

rows = []
for scen in scenarios:
    for q in quarters:
        as_of = q.to_timestamp(how="end").date()
        if scen == "BASELINE":
            gdp = rng.normal(2.0, 0.5)
            unemp = rng.normal(5.5, 0.4)
            rate = rng.normal(3.5, 0.5)
        elif scen == "ADVERSE":
            gdp = rng.normal(-1.0, 0.7)
            unemp = rng.normal(7.5, 0.7)
            rate = rng.normal(4.5, 0.7)
        else:
            gdp = rng.normal(-3.0, 1.0)
            unemp = rng.normal(10.0, 1.0)
            rate = rng.normal(6.0, 0.9)
        rows.append((scen, str(q), as_of, round(gdp,2), round(unemp,2), round(rate,2)))

src_macro_scenarios = pd.DataFrame(
    rows,
    columns=["scenario_name","period","as_of_date","gdp_yoy_pct","unemployment_pct","policy_rate_pct"]
)

src_customers.head(), src_loans.head(), src_transactions.head(), src_macro_scenarios.head()


### Persist Bronze Tables (Raw Landing)
Think of this like landing files into object storage + a raw schema for audit/replay.


In [ ]:
for t in ["bronze_customers","bronze_loans","bronze_transactions","bronze_macro_scenarios"]:
    conn.execute(f"DROP TABLE IF EXISTS {t}")

src_customers.to_sql("bronze_customers", conn, index=False)
src_loans.to_sql("bronze_loans", conn, index=False)
src_transactions.to_sql("bronze_transactions", conn, index=False)
src_macro_scenarios.to_sql("bronze_macro_scenarios", conn, index=False)

pd.read_sql("SELECT COUNT(*) AS n_transactions FROM bronze_transactions", conn)


## 2) Silver (Staging): Standardize + Validate
This is where analytics engineers enforce basic **data contracts**:
- required keys present
- date validity
- de-duplication (latest wins)
- standard types/labels


In [ ]:
bronze_customers = pd.read_sql("SELECT * FROM bronze_customers", conn, parse_dates=["origination_date","src_updated_at"])
bronze_loans = pd.read_sql("SELECT * FROM bronze_loans", conn, parse_dates=["origination_date","maturity_date","src_updated_at"])
bronze_tx = pd.read_sql("SELECT * FROM bronze_transactions", conn, parse_dates=["tx_date"])
bronze_macro = pd.read_sql("SELECT * FROM bronze_macro_scenarios", conn, parse_dates=["as_of_date"])

# --- Standardization ---
stg_customers = bronze_customers.copy()
stg_customers["legal_name"] = stg_customers["legal_name"].str.strip()
stg_customers["industry"] = stg_customers["industry"].str.title()
stg_customers["region"] = stg_customers["region"].str.upper()
stg_customers["current_rating"] = stg_customers["current_rating"].str.upper()

stg_loans = bronze_loans.copy()
stg_loans["product_type"] = stg_loans["product_type"].str.strip()
stg_loans["currency"] = stg_loans["currency"].str.upper()

stg_tx = bronze_tx.copy()
stg_tx["tx_type"] = stg_tx["tx_type"].str.upper()

# --- Quality checks (minimal demo) ---
assert stg_customers["customer_id"].isna().sum() == 0
assert stg_loans["loan_id"].isna().sum() == 0
assert stg_loans["customer_id"].isna().sum() == 0
assert (stg_loans["maturity_date"] >= stg_loans["origination_date"]).all()

# De-dupe by natural key using latest update timestamp
stg_customers = stg_customers.sort_values("src_updated_at").drop_duplicates(subset=["customer_id"], keep="last")
stg_loans = stg_loans.sort_values("src_updated_at").drop_duplicates(subset=["loan_id"], keep="last")

for t in ["silver_customers","silver_loans","silver_transactions","silver_macro_scenarios"]:
    conn.execute(f"DROP TABLE IF EXISTS {t}")

stg_customers.to_sql("silver_customers", conn, index=False)
stg_loans.to_sql("silver_loans", conn, index=False)
stg_tx.to_sql("silver_transactions", conn, index=False)
bronze_macro.to_sql("silver_macro_scenarios", conn, index=False)

pd.read_sql("SELECT COUNT(*) AS n_customers FROM silver_customers", conn)


## 3) Gold (Core): Canonical Entities + History + Snapshots
Core models enable point-in-time correctness and reproducible analytics.

We’ll build:
1) `gold_dim_customer_scd2` – **SCD Type 2** for rating changes
2) `gold_dim_loan` – loan entity with surrogate key
3) `gold_fact_loan_monthly_snapshot` – monthly balances (portfolio state)


### 3.1 Customer SCD Type 2 (Ratings over time)
In practice, you ingest customer rating changes as deltas; here we simulate some changes.


In [ ]:
silver_customers = pd.read_sql("SELECT * FROM silver_customers", conn, parse_dates=["origination_date","src_updated_at"])

# Simulate 0-2 rating changes per customer across 2024-2025
changes = []
for _, row in silver_customers.iterrows():
    cid = int(row["customer_id"])
    current = row["current_rating"]
    n_changes = int(rng.integers(0, 3))
    if n_changes == 0:
        continue
    ch_dates = sorted(rand_dates(n_changes, start=date(2024,1,1), end=date(2025,12,31)))
    for d in ch_dates:
        idx = ratings.index(current)
        step = int(rng.choice([-2,-1,1,2], p=[0.15,0.35,0.35,0.15]))
        idx2 = int(np.clip(idx + step, 0, len(ratings)-1))
        current = ratings[idx2]
        changes.append((cid, d, current))

rating_changes = pd.DataFrame(changes, columns=["customer_id","change_date","new_rating"]).sort_values(["customer_id","change_date"])
rating_changes.head()


In [ ]:
cust_base = silver_customers[["customer_id","legal_name","industry","region","income_band","origination_date","current_rating"]].copy()

scd_rows = []
for _, c in cust_base.iterrows():
    cid = int(c["customer_id"])
    eff_start = c["origination_date"].date() if hasattr(c["origination_date"], "date") else c["origination_date"]

    cust_changes = rating_changes[rating_changes["customer_id"] == cid]
    if cust_changes.empty:
        scd_rows.append({
            "customer_id": cid,
            "legal_name": c["legal_name"],
            "industry": c["industry"],
            "region": c["region"],
            "income_band": c["income_band"],
            "rating": c["current_rating"],
            "effective_start": eff_start,
            "effective_end": date(9999,12,31),
            "is_current": 1
        })
    else:
        # baseline version until first change
        first_change = cust_changes["change_date"].iloc[0]
        scd_rows.append({
            "customer_id": cid,
            "legal_name": c["legal_name"],
            "industry": c["industry"],
            "region": c["region"],
            "income_band": c["income_band"],
            "rating": c["current_rating"],
            "effective_start": eff_start,
            "effective_end": first_change,
            "is_current": 0
        })
        # versions from each change date forward
        for i, ch in cust_changes.reset_index(drop=True).iterrows():
            start = ch["change_date"]
            end = cust_changes["change_date"].iloc[i+1] if i+1 < len(cust_changes) else date(9999,12,31)
            scd_rows.append({
                "customer_id": cid,
                "legal_name": c["legal_name"],
                "industry": c["industry"],
                "region": c["region"],
                "income_band": c["income_band"],
                "rating": ch["new_rating"],
                "effective_start": start,
                "effective_end": end,
                "is_current": 1 if end == date(9999,12,31) else 0
            })

gold_dim_customer_scd2 = (pd.DataFrame(scd_rows)
                         .sort_values(["customer_id","effective_start"])
                         .reset_index(drop=True))
gold_dim_customer_scd2.insert(0, "customer_sk", np.arange(1, len(gold_dim_customer_scd2) + 1))

conn.execute("DROP TABLE IF EXISTS gold_dim_customer_scd2")
gold_dim_customer_scd2.to_sql("gold_dim_customer_scd2", conn, index=False)

gold_dim_customer_scd2.head(12)


### 3.2 Loan Entity
Create a canonical loan entity with a surrogate key.


In [ ]:
silver_loans = pd.read_sql("SELECT * FROM silver_loans", conn, parse_dates=["origination_date","maturity_date","src_updated_at"])

gold_dim_loan = silver_loans[[
    "loan_id","customer_id","product_type","currency","origination_date","maturity_date","original_principal","interest_rate"
]].copy()

gold_dim_loan = gold_dim_loan.sort_values("loan_id").reset_index(drop=True)
gold_dim_loan.insert(0, "loan_sk", np.arange(1, len(gold_dim_loan) + 1))

conn.execute("DROP TABLE IF EXISTS gold_dim_loan")
gold_dim_loan.to_sql("gold_dim_loan", conn, index=False)

gold_dim_loan.head()


### 3.3 Monthly Portfolio Snapshot
Compute **month-end balances** by accumulating transactions from original principal.


In [ ]:
silver_tx = pd.read_sql("SELECT * FROM silver_transactions", conn, parse_dates=["tx_date"])

month_ends = pd.date_range(start=start_dt, end=end_dt, freq="M").date

# transactions by month-end
silver_tx["month_end"] = silver_tx["tx_date"].dt.to_period("M").dt.to_timestamp("M").dt.date
tx_m = (silver_tx.groupby(["loan_id","month_end"], as_index=False)["amount"]
        .sum()
        .rename(columns={"amount":"net_tx_amount"}))

loan_principal = gold_dim_loan[["loan_id","original_principal"]].copy()

# full grid loan x month_end
grid = (pd.MultiIndex.from_product([loan_principal["loan_id"].unique(), month_ends], names=["loan_id","month_end"])
        .to_frame(index=False))

snap = grid.merge(loan_principal, on="loan_id", how="left").merge(tx_m, on=["loan_id","month_end"], how="left")
snap["net_tx_amount"] = snap["net_tx_amount"].fillna(0.0)

snap = snap.sort_values(["loan_id","month_end"])
snap["balance"] = snap.groupby("loan_id")["net_tx_amount"].cumsum() + snap["original_principal"]
snap["balance"] = snap["balance"].clip(lower=0)

gold_fact_loan_monthly_snapshot = snap[["loan_id","month_end","balance"]].copy()

conn.execute("DROP TABLE IF EXISTS gold_fact_loan_monthly_snapshot")
gold_fact_loan_monthly_snapshot.to_sql("gold_fact_loan_monthly_snapshot", conn, index=False)

gold_fact_loan_monthly_snapshot.head()


## 4) Risk Metrics Core Fact (PD/LGD/EAD) + Expected Loss
In real platforms, PD/LGD/EAD often come from model output tables.
Here we generate a simple synthetic model:
- PD depends on rating + unemployment
- LGD depends on product type (secured vs unsecured proxy)
- EAD approximated as balance
Expected Loss = PD × LGD × EAD


In [ ]:
macro_baseline = pd.read_sql(
    "SELECT * FROM silver_macro_scenarios WHERE scenario_name='BASELINE'",
    conn,
    parse_dates=["as_of_date"]
)
macro_q = macro_baseline[["period","unemployment_pct"]].copy()

rating_to_pd = {"AAA": 0.002, "AA": 0.004, "A": 0.008, "BBB": 0.02, "BB": 0.05, "B": 0.12, "CCC": 0.25}
product_to_lgd = {"Mortgage": 0.20, "AutoLoan": 0.35, "TermLoan": 0.45, "Revolver": 0.55}

# Time dimension at month-end
dim_time = pd.DataFrame({"date": pd.to_datetime(list(month_ends))})
dim_time["date_key"] = dim_time["date"].dt.strftime("%Y%m%d").astype(int)
dim_time["month_end"] = dim_time["date"].dt.date
dim_time["quarter"] = dim_time["date"].dt.to_period("Q").astype(str)
dim_time["year"] = dim_time["date"].dt.year
dim_time["month"] = dim_time["date"].dt.month

# Join snapshot to macro quarter
snap = gold_fact_loan_monthly_snapshot.merge(dim_time[["month_end","quarter"]], on="month_end", how="left")
snap = snap.merge(macro_q, left_on="quarter", right_on="period", how="left")

# Attach loan + customer
loan = gold_dim_loan[["loan_id","customer_id","product_type"]].copy()
snap2 = snap.merge(loan, on="loan_id", how="left")

# SCD2 join to get correct rating for that month_end
cust_scd2 = gold_dim_customer_scd2.copy()
cust_scd2["effective_start"] = pd.to_datetime(cust_scd2["effective_start"])
cust_scd2["effective_end"] = pd.to_datetime(cust_scd2["effective_end"])

snap2["as_of"] = pd.to_datetime(snap2["month_end"])
snap2 = snap2.merge(
    cust_scd2[["customer_sk","customer_id","rating","effective_start","effective_end"]],
    on="customer_id",
    how="left"
)
snap2 = snap2[(snap2["as_of"] >= snap2["effective_start"]) & (snap2["as_of"] < snap2["effective_end"])]

# Compute synthetic PD/LGD/EAD
base_pd = snap2["rating"].map(rating_to_pd).astype(float)
unemp = snap2["unemployment_pct"].fillna(6.0).astype(float)
pd_adj = base_pd * (1 + 0.08 * (unemp - 5.5))  # unemployment above baseline increases PD

lgd = snap2["product_type"].map(product_to_lgd).astype(float)
ead = snap2["balance"].astype(float)

risk_metrics = snap2[["loan_id","month_end","customer_sk","product_type"]].copy()
risk_metrics["pd"] = pd_adj.clip(0, 1)
risk_metrics["lgd"] = lgd.clip(0, 1)
risk_metrics["ead"] = ead
risk_metrics["expected_loss"] = risk_metrics["pd"] * risk_metrics["lgd"] * risk_metrics["ead"]

conn.execute("DROP TABLE IF EXISTS gold_fact_risk_metrics")
risk_metrics.to_sql("gold_fact_risk_metrics", conn, index=False)

risk_metrics.head()


## 5) Mart Layer: Star Schema (Facts + Dims)
This is what downstream analytics and reporting teams want: simple, fast joins.
- `mart_dim_time`
- `mart_dim_customer` (current view)
- `mart_dim_loan`
- `mart_fact_exposure`
- `mart_fact_risk_metrics`


In [ ]:
# dim_time
conn.execute("DROP TABLE IF EXISTS mart_dim_time")
dim_time.to_sql("mart_dim_time", conn, index=False)

# dim_customer (current view from SCD2)
cust_current = pd.read_sql("SELECT * FROM gold_dim_customer_scd2 WHERE is_current=1", conn)
conn.execute("DROP TABLE IF EXISTS mart_dim_customer")
cust_current.to_sql("mart_dim_customer", conn, index=False)

# dim_loan
conn.execute("DROP TABLE IF EXISTS mart_dim_loan")
gold_dim_loan.to_sql("mart_dim_loan", conn, index=False)

# fact_exposure (monthly)
fact_exposure = gold_fact_loan_monthly_snapshot.merge(
    gold_dim_loan[["loan_id","loan_sk","customer_id"]],
    on="loan_id",
    how="left"
)
fact_exposure = fact_exposure.merge(dim_time[["month_end","date_key"]], on="month_end", how="left")
fact_exposure = fact_exposure[["date_key","loan_sk","loan_id","customer_id","balance"]].rename(columns={"balance":"exposure_balance"})

conn.execute("DROP TABLE IF EXISTS mart_fact_exposure")
fact_exposure.to_sql("mart_fact_exposure", conn, index=False)

# fact_risk_metrics (monthly)
fact_rm = risk_metrics.merge(gold_dim_loan[["loan_id","loan_sk","customer_id"]], on="loan_id", how="left")
fact_rm = fact_rm.merge(dim_time[["month_end","date_key"]], on="month_end", how="left")
fact_rm = fact_rm[["date_key","loan_sk","loan_id","customer_id","customer_sk","pd","lgd","ead","expected_loss"]]

conn.execute("DROP TABLE IF EXISTS mart_fact_risk_metrics")
fact_rm.to_sql("mart_fact_risk_metrics", conn, index=False)

pd.read_sql("SELECT * FROM mart_fact_risk_metrics LIMIT 5", conn)


## 6) Stress Testing Output Fact (Scenario-Keyed Results)
A typical deliverable is a table keyed by `(scenario, period, exposure)` that enables scenario comparison.

We compute quarterly expected loss under each scenario by:
- taking quarter-end exposure per loan
- joining correct customer rating at quarter-end (SCD2)
- applying scenario unemployment to adjust PD


In [ ]:
macro_all = pd.read_sql("SELECT * FROM silver_macro_scenarios", conn, parse_dates=["as_of_date"]).rename(columns={"scenario_name":"scenario"})

# Quarter-end exposure proxy: pick last month-end row per loan per quarter
fact_exposure_q = fact_exposure.merge(dim_time[["date_key","quarter"]], on="date_key", how="left")
fact_exposure_q["month_end_ts"] = pd.to_datetime(fact_exposure_q["date_key"].astype(str), format="%Y%m%d")

fact_exposure_q = (fact_exposure_q.sort_values(["loan_id","quarter","month_end_ts"])
                   .groupby(["loan_id","quarter"], as_index=False)
                   .tail(1))

# Quarter end timestamps for SCD2 join
quarter_end = dim_time.groupby("quarter")["month_end"].max().reset_index()
quarter_end["as_of"] = pd.to_datetime(quarter_end["month_end"])

base = fact_exposure_q.merge(quarter_end[["quarter","as_of"]], on="quarter", how="left")
base = base.merge(gold_dim_loan[["loan_id","customer_id","product_type"]], on="loan_id", how="left")

cust_scd2 = gold_dim_customer_scd2.copy()
cust_scd2["effective_start"] = pd.to_datetime(cust_scd2["effective_start"])
cust_scd2["effective_end"] = pd.to_datetime(cust_scd2["effective_end"])

tmp = base.merge(
    cust_scd2[["customer_sk","customer_id","rating","effective_start","effective_end"]],
    on="customer_id",
    how="left"
)
tmp = tmp[(tmp["as_of"] >= tmp["effective_start"]) & (tmp["as_of"] < tmp["effective_end"])]

# Join macro scenario by quarter period
tmp = tmp.merge(
    macro_all[["scenario","period","unemployment_pct"]],
    left_on="quarter",
    right_on="period",
    how="left"
)

base_pd = tmp["rating"].map(rating_to_pd).astype(float)
unemp = tmp["unemployment_pct"].astype(float)
pd_scen = base_pd * (1 + 0.08 * (unemp - 5.5))
lgd = tmp["product_type"].map(product_to_lgd).astype(float)
ead = tmp["exposure_balance"].astype(float)

tmp["scenario_pd"] = pd_scen.clip(0, 1)
tmp["scenario_lgd"] = lgd.clip(0, 1)
tmp["scenario_ead"] = ead
tmp["scenario_expected_loss"] = tmp["scenario_pd"] * tmp["scenario_lgd"] * tmp["scenario_ead"]

stress = tmp[["scenario","quarter","loan_id","customer_id","customer_sk","scenario_pd","scenario_lgd","scenario_ead","scenario_expected_loss"]].copy()

conn.execute("DROP TABLE IF EXISTS mart_fact_stress_expected_loss")
stress.to_sql("mart_fact_stress_expected_loss", conn, index=False)

stress.head()


## 7) Example Analytics Queries (What the Model Enables)
These are the kinds of questions your mart supports.


In [ ]:
# 7.1 Portfolio exposure by product and quarter
q1 = '''
SELECT t.quarter,
       l.product_type,
       SUM(e.exposure_balance) AS total_exposure
FROM mart_fact_exposure e
JOIN mart_dim_time t ON e.date_key = t.date_key
JOIN mart_dim_loan l ON e.loan_sk = l.loan_sk
GROUP BY t.quarter, l.product_type
ORDER BY t.quarter, total_exposure DESC
LIMIT 20
'''
pd.read_sql(q1, conn)


In [ ]:
# 7.2 Expected loss by rating (baseline monthly core fact)
q2 = '''
SELECT c.rating,
       AVG(r.expected_loss) AS avg_expected_loss,
       SUM(r.expected_loss) AS total_expected_loss,
       COUNT(*) AS n_rows
FROM mart_fact_risk_metrics r
JOIN gold_dim_customer_scd2 c ON r.customer_sk = c.customer_sk
GROUP BY c.rating
ORDER BY total_expected_loss DESC
'''
pd.read_sql(q2, conn)


In [ ]:
# 7.3 Stress impact: baseline vs severely adverse by quarter
q3 = '''
SELECT quarter,
       SUM(CASE WHEN scenario='BASELINE' THEN scenario_expected_loss ELSE 0 END) AS el_baseline,
       SUM(CASE WHEN scenario='SEVERELY_ADVERSE' THEN scenario_expected_loss ELSE 0 END) AS el_severe,
       (SUM(CASE WHEN scenario='SEVERELY_ADVERSE' THEN scenario_expected_loss ELSE 0 END) -
        SUM(CASE WHEN scenario='BASELINE' THEN scenario_expected_loss ELSE 0 END)) AS delta_el
FROM mart_fact_stress_expected_loss
GROUP BY quarter
ORDER BY quarter
'''
pd.read_sql(q3, conn)


## 8) Interview Mapping (How to Explain This)
A clean way to describe the pipeline:

- **Bronze:** raw landing for auditability/replay
- **Silver:** standardize + validate + de-dupe (data contracts)
- **Gold:** core entities + **SCD2** (point-in-time correctness) + **snapshots** (portfolio state)
- **Marts:** star schema for analytics/reporting
- **Stress outputs:** scenario-keyed facts for comparisons and regulatory reporting

Key modeling patterns used: **SCD Type 2**, **snapshots**, **star schema**, **scenario result facts**.


In [ ]:
conn.commit()
conn.close()
print("Done. SQLite DB written to:", DB_PATH)
